In [ ]:
# src/config.py
import os
from dotenv import load_dotenv

# Carrega o .env UMA vez, no ponto central do projeto.
load_dotenv()

# ID do modelo padrão — trocável em um só lugar.
# (lembre: o ID muda, mas a API do LangChain não)
MODEL_ID = os.getenv("MODEL_ID", "openai:gpt-4o-mini")

# Modelo de embeddings (usado no RAG do Módulo 2)
EMBEDDINGS_ID = os.getenv("EMBEDDINGS_ID", "openai:text-embedding-3-small")

# As chaves NÃO ficam no código — vêm do ambiente (.env).
# O SDK da OpenAI lê OPENAI_API_KEY automaticamente; aqui só validamos.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Defina OPENAI_API_KEY no seu .env (veja .env.example)")

In [ ]:
# src/agents/assistente.py
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

from src.config import MODEL_ID          # <- vem do config, não hard-coded

model = init_chat_model(MODEL_ID)        # troca de modelo? só mexe no config/.env
agent = create_agent(model, tools=[])

In [ ]:
%pip install -U pytest

In [ ]:
# tests/test_tools.py
from src.tools.calculadora import somar   # a peça isolada que queremos testar

def test_somar_positivos():
    assert somar(2, 3) == 5

def test_somar_negativos():
    assert somar(-1, -1) == -2

In [ ]:
# tests/test_prompts.py
from src.prompts.resumo import prompt_resumo   # um ChatPromptTemplate do projeto

def test_variaveis_esperadas():
    # protege contra alguém renomear/remover uma variável sem querer
    assert set(prompt_resumo.input_variables) == {"tema", "contexto"}

def test_preenche_os_valores():
    msgs = prompt_resumo.format_messages(tema="LangChain", contexto="v1 lançada")
    conteudo = msgs[-1].content
    assert "LangChain" in conteudo       # o valor foi injetado no texto?
    assert "v1 lançada" in conteudo

In [ ]:
# tests/test_chains.py
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser

from src.prompts.resumo import prompt_resumo

def test_chain_converte_saida_em_string():
    # modelo falso: sempre "responde" isto, de forma determinística
    fake = GenericFakeChatModel(messages=iter([AIMessage(content="Resumo: LangChain v1.")]))

    chain = prompt_resumo | fake | StrOutputParser()
    saida = chain.invoke({"tema": "LangChain", "contexto": "v1"})

    # NÃO testamos o TEOR da resposta (isso é papel do modelo),
    # e sim que a CHAIN montou e o parser entregou uma string.
    assert isinstance(saida, str)
    assert saida.startswith("Resumo:")   # veio do fake, é previsível

In [ ]:
# na raiz do projeto (app/)
pytest                  # roda todos os testes de tests/
pytest -v               # modo verboso (mostra cada teste)
pytests/test_tool.py    #roda um arquivo só